### Install

In [1]:
pip install ta yfinance

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=3ce8baaf11681f1826d5dba5cabb6e8cb99286d89efefc62a0b8d48b1fb68c2b
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
from datetime import datetime, timedelta
from pandas.tseries.offsets import BDay
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.collections import LineCollection
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)



### Derivates and Trading Signals

In [3]:
def calculate_first_derivative(series, window=3):
    """
    First derivative: rate of change (velocity).
    Positive = upward momentum, Negative = downward momentum.
    Flips identify early trend reversals.
    """
    return series.diff(window) / window

def calculate_second_derivative(series, window=3):
    """
    Second derivative: acceleration/curvature.
    Positive = accelerating up (parabolic rise)
    Negative = accelerating down (crash)
    Sign flips = inflection points (momentum changes)
    """
    first_deriv = calculate_first_derivative(series, window)
    second_deriv = first_deriv.diff(window) / window
    return second_deriv

def calculate_third_derivative(series, window=3):
    """
    Third derivative: jerk (rate of acceleration change).
    Detects parabolic extremes and volatility bursts.
    Extreme values = market is becoming unstable/overextended.
    """
    second_deriv = calculate_second_derivative(series, window)
    third_deriv = second_deriv.diff(window) / window
    return third_deriv

def detect_inflection_points(series, window=3):
    """
    Detects where second derivative crosses zero.
    These are momentum inflection points:
    - Negative to Positive = trend acceleration begins (potential buy)
    - Positive to Negative = trend deceleration (potential sell/reduce)
    """
    second_deriv = calculate_second_derivative(series, window)
    # Find sign changes
    sign_changes = np.sign(second_deriv).diff()
    inflections = sign_changes.fillna(0) != 0
    return inflections, second_deriv

def calculate_curvature_intensity(series, window=3):
    """
    Magnitude of second derivative normalized to price.
    Higher = sharper turns = more explosive moves.
    Used to weight signal strength.
    """
    second_deriv = calculate_second_derivative(series, window)
    # Normalize by rolling price mean to scale-adjust
    price_mean = series.rolling(window * 2).mean()
    curvature_intensity = second_deriv.abs() / (price_mean + 1e-10)
    return curvature_intensity

def calculate_momentum_divergence(price, momentum_indicator, window=5):
    """
    Detects divergence between price and momentum (RSI, MACD, etc).
    Price makes new high but momentum doesn't = weakening upside (sell signal)
    Price makes new low but momentum doesn't = bottoming (buy signal)
    """
    # Rolling max/min
    price_max = price.rolling(window).max()
    price_min = price.rolling(window).min()
    mom_max = momentum_indicator.rolling(window).max()
    mom_min = momentum_indicator.rolling(window).min()

    # Detect divergences
    bearish_div = (price > price_max.shift(1)) & (momentum_indicator < mom_max.shift(1))
    bullish_div = (price < price_min.shift(1)) & (momentum_indicator > mom_min.shift(1))

    return bearish_div, bullish_div

def is_parabolic(series, window=20, threshold=0.5):
    """
    Detects parabolic moves using exponential acceleration.
    If third derivative (jerk) is extreme = parabolic overextension.
    Used to trigger position reduction (profit-taking).
    """
    third_deriv = calculate_third_derivative(series, window=3)
    # Normalize jerk
    jerk_zscore = (third_deriv - third_deriv.rolling(window).mean()) / (third_deriv.rolling(window).std() + 1e-10)
    is_parabolic_up = jerk_zscore > threshold
    is_parabolic_down = jerk_zscore < -threshold
    return is_parabolic_up, is_parabolic_down

In [4]:
def calculate_signal_strength(
    price_deriv_1,      # 1st deriv of price
    price_deriv_2,      # 2nd deriv of price
    rsi,                # RSI oscillator
    rsi_deriv_1,        # 1st deriv of RSI
    inflection,         # Is at inflection point?
    parabolic_up,       # Is price parabolic?
    parabolic_down,
    volume_surge=None   # Optional: volume indicator
):
    """
    Calculates signal strength (0.0 to 1.0) based on convergence of indicators.
    Higher strength = more aggressive leverage.

    0.0-0.33 = Hold/1x (weak signal)
    0.33-0.66 = 2x Leverage (strong signal)
    0.66-1.0 = 3x Leverage (extreme signal + confluence)
    """
    score = 0.0
    weights_sum = 0.0

    # 1st derivative: positive = upward momentum (0.1 weight)
    if pd.notna(price_deriv_1):
        momentum_score = np.clip(price_deriv_1 / (abs(price_deriv_1) + 1e-10), -1, 1)
        score += max(0, momentum_score) * 0.15
        weights_sum += 0.15

    # 2nd derivative: inflection point from negative to positive = acceleration (0.2 weight)
    if pd.notna(price_deriv_2) and inflection:
        if price_deriv_2 > 0:  # Transitioning to upward acceleration
            score += 0.25
        weights_sum += 0.25

    # RSI: reward extremes, punish overbought (0.2 weight)
    if pd.notna(rsi):
        if rsi < 30:  # Oversold = buy opportunity
            rsi_score = (30 - rsi) / 30  # Closer to 0 = stronger
            score += rsi_score * 0.20
        elif rsi > 70:  # Overbought = reduce/sell
            score -= (rsi - 70) / 30 * 0.15
        weights_sum += 0.20

    # RSI 1st derivative: RSI rising = momentum building (0.15 weight)
    if pd.notna(rsi_deriv_1):
        rsi_momentum_score = np.clip(rsi_deriv_1 / (abs(rsi_deriv_1) + 1e-10), -1, 1)
        score += max(0, rsi_momentum_score) * 0.15
        weights_sum += 0.15

    # Parabolic up: boost confidence if moving parabolic up (0.15 weight)
    if parabolic_up:
        score += 0.15
        weights_sum += 0.15

    # Parabolic down: severely penalize (reduce position) (0.2 weight)
    if parabolic_down:
        score -= 0.25
        weights_sum += 0.20

    # Volume surge: if provided (0.1 weight)
    if volume_surge is not None and volume_surge:
        score += 0.10
        weights_sum += 0.10

    # Normalize to 0-1
    if weights_sum > 0:
        normalized_score = np.clip(score / weights_sum, -1.0, 1.0)
        # Shift to 0-1 range
        signal_strength = (normalized_score + 1.0) / 2.0
    else:
        signal_strength = 0.5

    return np.clip(signal_strength, 0.0, 1.0)

def determine_leverage_level(signal_strength, rsi, price_parabolic_up, price_parabolic_down):
    """
    Maps signal strength + oscillator extremes to leverage level.

    Returns:
    - 'Hold' (0x): No position or reduce
    - 'Buy' (1x): Moderate signal
    - 'Buy_2x' (2x): Strong signal
    - 'Buy_3x' (3x): Extreme signal + inflection
    """
    # If parabolic down AND high RSI, reduce position aggressively
    if price_parabolic_down and rsi > 70:
        return 'Reduce', 0.5  # Exit half

    if price_parabolic_down and rsi > 80:
        return 'Sell', 0.0  # Exit full

    # If parabolic up AND oversold RSI, maximum leverage
    if price_parabolic_up and rsi < 30:
        return 'Buy_3x', 3.0

    # If not parabolic but strong inflection signal
    if signal_strength > 0.70:
        return 'Buy_3x', 3.0
    elif signal_strength > 0.55:
        return 'Buy_2x', 2.0
    elif signal_strength > 0.40:
        return 'Buy', 1.0
    elif signal_strength > 0.30:
        return 'Hold', 1.0
    else:
        return 'Reduce', 0.5

In [5]:
def add_calculus_indicators(df):
    """
    Adds all derivative-based and advanced indicators to DataFrame.
    """
    df = df.copy()

    if 'Close' not in df.columns:
        raise ValueError("DataFrame must contain 'Close' column.")

    # --- Base EMAs ---
    df['EMA50'] = ta.trend.ema_indicator(df['Close'], window=50)
    df['EMA200'] = ta.trend.ema_indicator(df['Close'], window=200)
    df['EMA20'] = ta.trend.ema_indicator(df['Close'], window=20)

    # --- Oscillators ---
    df['RSI'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()
    df['RSI2'] = ta.momentum.RSIIndicator(close=df['Close'], window=2).rsi()
    df['RSI_Fast'] = ta.momentum.RSIIndicator(close=df['Close'], window=5).rsi()

    # --- MACD for additional momentum ---
    macd = ta.trend.MACD(close=df['Close'], window_fast=12, window_slow=26, window_sign=9)
    df['MACD'] = macd.macd()
    df['MACD_Signal'] = macd.macd_signal()
    df['MACD_Diff'] = macd.macd_diff()

    # --- All Time High metrics ---
    df['AllTimeHigh'] = df['Close'].cummax()
    df['OffATH'] = (df['Close'] - df['AllTimeHigh']) / df['AllTimeHigh']

    # --- Bollinger Bands ---
    df['BB_middle'] = ta.trend.sma_indicator(df['Close'], window=20)
    df['BB_upper'] = df['BB_middle'] + 2 * df['Close'].rolling(20).std()
    df['BB_lower'] = df['BB_middle'] - 2 * df['Close'].rolling(20).std()

    # ===== CALCULUS INDICATORS =====

    # Price derivatives (on close price)
    df['Price_d1'] = calculate_first_derivative(df['Close'], window=3)
    df['Price_d2'] = calculate_second_derivative(df['Close'], window=3)
    df['Price_d3'] = calculate_third_derivative(df['Close'], window=3)

    # EMA50 derivatives (smoother than raw price)
    df['EMA50_d1'] = calculate_first_derivative(df['EMA50'], window=3)
    df['EMA50_d2'] = calculate_second_derivative(df['EMA50'], window=3)
    df['EMA50_d3'] = calculate_third_derivative(df['EMA50'], window=3)

    # RSI derivatives (momentum of momentum)
    df['RSI_d1'] = calculate_first_derivative(df['RSI'], window=3)
    df['RSI_d2'] = calculate_second_derivative(df['RSI'], window=3)

    # MACD derivatives (for earlier signals)
    df['MACD_d1'] = calculate_first_derivative(df['MACD'], window=3)

    # Inflection point detection
    df['Inflection'], df['Price_d2_raw'] = detect_inflection_points(df['Close'], window=3)

    # Curvature intensity
    df['Curvature_Intensity'] = calculate_curvature_intensity(df['Close'], window=3)

    # Parabolic detection
    df['Parabolic_Up'], df['Parabolic_Down'] = is_parabolic(df['Close'], window=20, threshold=0.5)

    # Momentum divergence
    df['Bearish_Div'], df['Bullish_Div'] = calculate_momentum_divergence(df['Close'], df['RSI'], window=5)

    # Normalize third derivative for extreme detection
    df['Price_d3_zscore'] = (df['Price_d3'] - df['Price_d3'].rolling(20).mean()) / (df['Price_d3'].rolling(20).std() + 1e-10)

    # Volume rate of change
    if 'Volume' in df.columns:
        df['Volume_d1'] = df['Volume'].pct_change()
        df['Volume_MA'] = df['Volume'].rolling(20).mean()
        df['Volume_Surge'] = df['Volume'] > df['Volume_MA'] * 1.5
    else:
        df['Volume_Surge'] = False

    # Forward fill to handle initial NaNs
    df = df.ffill()

    return df

In [6]:
def apply_derivative_signal_logic(data, start_date=None, is_index=False):
    """
    Advanced signal logic using calculus and inflection points.

    FIXES FOR OVERFITTING:
    1. Trend filter: Don't exit during bull trends (EMA50 > EMA200)
    2. Volume confirmation: RSI exit requires low volume (not high volume rallies)
    3. d3 only for panic buys: High jerk in downtrend = buy, not sell
    4. Separate overbought from exhaustion: Need trend reversal + overbought
    5. No cascading confirmations: Signal fires once per state change
    """
    data = data.copy()
    signals = []
    leverage_levels = []

    start_index = data['EMA50_d1'].first_valid_index()
    start_i = data.index.get_loc(start_index)

    for i in range(len(data)):
        row = data.iloc[i]

        if i < start_i or np.isnan(row['EMA50_d1']):
            signals.append(None)
            leverage_levels.append(0)
            continue

        # ===== TREND IDENTIFICATION =====
        ema50_above_200 = row['EMA50'] > row['EMA200']
        ema50_rising = row['EMA50_d1'] > 0
        ema200_rising = data['EMA200'].iloc[i-5:i].mean() < row['EMA200']

        bull_trend = ema50_above_200 and ema50_rising and ema200_rising
        bear_trend = not ema50_above_200  # Simple: below EMA200 = bear

        # ===== INFLECTION POINT DETECTION =====
        price_d2_today = row['Price_d2']
        price_d2_yesterday = data['Price_d2'].iloc[i-1] if i > 0 else np.nan

        inflection_buy = False
        if pd.notna(price_d2_today) and pd.notna(price_d2_yesterday):
            if price_d2_yesterday < 0 and price_d2_today > 0:
                inflection_buy = True

        # ===== MOMENTUM CONVERGENCE =====
        price_momentum = row['Price_d1'] > 0
        ema50_momentum = row['EMA50_d1'] > 0

        rsi_oversold = row['RSI'] < 35
        rsi_rising = row['RSI_d1'] > 0
        rsi_oversold_extreme = row['RSI2'] < 10

        macd_bullish = row['MACD'] > row['MACD_Signal']
        macd_momentum = row['MACD_d1'] > 0

        # ===== VOLUME CHECKS =====
        volume_surge = row['Volume_Surge'] if 'Volume_Surge' in row.index else False
        volume_declining = False
        if i > 20:
            recent_vol = data['Volume'].iloc[i-5:i].mean()
            vol_20day = data['Volume'].iloc[i-20:i].mean()
            volume_declining = recent_vol < vol_20day * 0.85  # Below 85% of 20-day avg

        # ===== BUY SIGNALS (UNCHANGED) =====

        early_buy = (
            inflection_buy and
            row['RSI'] < 50 and
            ema50_momentum and
            bull_trend
        )

        standard_buy = (
            bull_trend and
            price_momentum and
            rsi_rising and
            (rsi_oversold or macd_bullish) and
            row['Close'] > row['EMA50']
        )

        # ===== SELL SIGNALS (NEW WITH TREND FILTER) =====

        # FIX 1: Don't exit just because overbought in a bull trend
        # Only exit overbought if TREND HAS REVERSED
        rsi_overbought_in_downtrend = (
            bear_trend and  # Trend must be down
            row['RSI'] > 70 and
            row['Price_d1'] < 0  # Price also falling
        )

        # FIX 2: Volume-confirmed overbought exhaustion
        # High RSI + LOW volume = weak rally = exit
        # High RSI + HIGH volume = strong rally = hold
        exhaustion_sell = (
            bear_trend and
            row['RSI'] > 75 and
            volume_declining and  # FIX: Add volume check
            row['Price_d1'] < 0
        )

        # FIX 3: Use d3 ONLY for panic buys in downtrends, not exits
        # In uptrends, d3 extreme = acceleration, not exhaustion
        panic_exit = (
            row['Parabolic_Down'] and  # d3 < -1.5
            bear_trend and  # Must be in downtrend
            row['RSI'] > 60  # And still have some momentum down
        )

        # FIX 4: Trend reversal + overbought = real exhaustion
        # Require both: trend flip + momentum fading
        trend_flip_sell = (
            row['EMA50'] < row['EMA200'] and  # EMA50 crossed below EMA200
            data['EMA50'].iloc[i-1] > data['EMA200'].iloc[i-1] and  # Was above yesterday
            row['RSI'] > 50  # Still have some momentum
        )

        # FIX 5: Bearish divergence in downtrend only
        divergence_sell = (
            row['Bearish_Div'] and
            bear_trend and
            row['Price_d1'] < 0 and
            row['RSI'] > 50
        )

        # ===== CALCULATE SIGNAL STRENGTH (MODIFIED) =====
        # Boost for buys, penalize more heavily for sells
        signal_strength = calculate_signal_strength(
            price_deriv_1=row['Price_d1'],
            price_deriv_2=row['Price_d2'],
            rsi=row['RSI'],
            rsi_deriv_1=row['RSI_d1'],
            inflection=row['Inflection'],
            parabolic_up=row['Parabolic_Up'] and bear_trend,  # FIX: Only in downtrend
            parabolic_down=row['Parabolic_Down'],
            volume_surge=volume_surge
        )

        # ===== DETERMINE LEVERAGE (MODIFIED) =====
        # Don't escalate to 3x unless trend is very strong
        leverage_action, leverage = determine_leverage_level(
            signal_strength,
            row['RSI'],
            row['Parabolic_Up'] and bear_trend,  # Only if downtrend
            row['Parabolic_Down']
        )

        # ===== FINAL SIGNAL ASSIGNMENT =====
        signal = None

        # SELLS (most restrictive first)
        if panic_exit:
            signal = 'Sell'
            leverage = 0.0
        elif trend_flip_sell:
            signal = 'Sell'
            leverage = 0.0
        elif divergence_sell:
            signal = 'Sell'
            leverage = 0.0
        elif exhaustion_sell:
            signal = 'Reduce'
            leverage = 0.5
        elif rsi_overbought_in_downtrend:
            signal = 'Reduce'
            leverage = 0.5

        # BUYS (only if not in any sell condition)
        elif early_buy and leverage_action == 'Buy_3x' and bull_trend:
            signal = 'Buy_3x (Inflection)'
            leverage = 3.0
        elif early_buy and leverage_action == 'Buy_2x' and bull_trend:
            signal = 'Buy_2x (Inflection)'
            leverage = 2.0
        elif early_buy and bull_trend:
            signal = 'Buy (Inflection)'
            leverage = 1.0
        elif leverage_action == 'Buy_3x' and bull_trend:
            signal = 'Buy_3x'
            leverage = 3.0
        elif leverage_action == 'Buy_2x' and bull_trend:
            signal = 'Buy_2x'
            leverage = 2.0
        elif standard_buy:
            signal = 'Buy'
            leverage = 1.0
        else:
            signal = 'Hold'
            leverage = 1.0

        signals.append(signal)
        leverage_levels.append(leverage)

    data['Signal'] = signals
    data['Leverage'] = leverage_levels

    # FIX 5: Only keep first signal and changes (no cascading fills)
    # Replace with NaN when signal repeats
    data = data.copy()
    data['Signal'] = signals
    data['Leverage'] = leverage_levels

    # Deduplicate: where signal equals previous, set to None
    data.loc[data['Signal'] == data['Signal'].shift(), 'Signal'] = None
    data['Signal'] = data['Signal'].ffill()

    if start_date:
        data = data[data.index >= start_date]

    data = data.dropna(subset=['Signal'])

    return data

### Calculate Returns

In [7]:
def calc_leveraged_returns(data, cash_rate=0.03):
    """
    Calculates returns with dynamic leverage based on signals.
    Uses 'Leverage' column to scale returns accordingly.
    """
    data = data.copy()

    daily_cash_rate = (1 + cash_rate) ** (1/252) - 1
    data['Returns'] = data['Close'].pct_change()
    data['BuyHold'] = (1 + data['Returns'].fillna(0)).cumprod()

    # Apply leverage from signal
    data['Leverage_Applied'] = data['Leverage'].shift()  # Use yesterday's leverage decision

    # Default to 1x (hold)
    data['Daily_ret'] = data['Returns'] * data['Leverage_Applied'].fillna(1.0)

    # For 'Sell' signals, use cash rate instead
    data.loc[data['Signal'].shift() == 'Sell', 'Daily_ret'] = daily_cash_rate
    data.loc[data['Signal'].shift() == 'Reduce', 'Daily_ret'] = data['Returns'] * 0.5

    data['Strat_ret'] = (1 + data['Daily_ret'].fillna(0)).cumprod()

    return data

def calculate_metrics(df, risk_free_rate=0.03):
    """
    Computes performance metrics for strategy vs. buy-and-hold.
    """
    df = df.copy()

    daily_rf = (1 + risk_free_rate) ** (1 / 252) - 1
    strat_daily = df['Daily_ret'].dropna()
    buyhold_daily = df['Returns'].dropna()

    metrics = {}

    def compute_stats(returns, label):
        cumulative = (1 + returns).prod() - 1
        if len(returns) > 0:
            annualized_return = (1 + cumulative) ** (252 / len(returns)) - 1
        else:
            annualized_return = 0
        annualized_vol = returns.std() * np.sqrt(252)
        sharpe = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol > 0 else np.nan
        cumulative_curve = (1 + returns).cumprod()
        peak = cumulative_curve.cummax()
        drawdown = (cumulative_curve - peak) / peak
        max_drawdown = drawdown.min()

        return {
            f'{label} Cumulative Return': cumulative,
            f'{label} Annualized Return': annualized_return,
            f'{label} Annualized Volatility': annualized_vol,
            f'{label} Sharpe Ratio': sharpe,
            f'{label} Max Drawdown': max_drawdown
        }

    metrics.update(compute_stats(strat_daily, 'Strategy'))
    metrics.update(compute_stats(buyhold_daily, 'BuyHold'))

    return metrics

### Plots (code)

In [8]:
def plot_derivatives(data, ticker='TICKER'):
    """
    Plots price + derivatives to visualize inflection points.
    """
    fig, axes = plt.subplots(4, 1, figsize=(16, 12))

    # Price + EMAs
    ax = axes[0]
    ax.plot(data.index, data['Close'], label='Close', color='black', linewidth=2)
    ax.plot(data.index, data['EMA50'], label='EMA50', color='blue', linestyle='--', alpha=0.7)
    ax.plot(data.index, data['EMA200'], label='EMA200', color='red', linestyle='--', alpha=0.7)
    ax.fill_between(data.index, data['BB_lower'], data['BB_upper'], alpha=0.1, color='gray')
    ax.set_title(f'{ticker} - Price & Trend Lines', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 1st Derivative (Momentum)
    ax = axes[1]
    colors_d1 = ['green' if x > 0 else 'red' for x in data['Price_d1']]
    ax.bar(data.index, data['Price_d1'], color=colors_d1, alpha=0.6)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title('1st Derivative - Price Momentum (Velocity)', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

    # 2nd Derivative (Acceleration) with inflection points
    ax = axes[2]
    colors_d2 = ['green' if x > 0 else 'red' for x in data['Price_d2']]
    ax.bar(data.index, data['Price_d2'], color=colors_d2, alpha=0.6, label='2nd Derivative')
    ax.axhline(0, color='black', linewidth=0.5)
    # Mark inflection points
    inflection_points = data[data['Inflection']]
    ax.scatter(inflection_points.index, inflection_points['Price_d2'], color='lime', s=100,
               marker='*', label='Inflection Point', zorder=5)
    ax.set_title('2nd Derivative - Acceleration (Concavity)', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3rd Derivative (Jerk) - Parabolic Detection
    ax = axes[3]
    colors_d3 = ['darkgreen' if x > 1 else 'red' if x < -1 else 'gray' for x in data['Price_d3_zscore']]
    ax.bar(data.index, data['Price_d3_zscore'], color=colors_d3, alpha=0.6, label='3rd Derivative (Z-Score)')
    ax.axhline(0.5, color='orange', linewidth=1, linestyle='--', label='Parabolic Threshold')
    ax.axhline(-0.5, color='orange', linewidth=1, linestyle='--')
    parabolic_up = data[data['Parabolic_Up']]
    ax.scatter(parabolic_up.index, parabolic_up['Price_d3_zscore'], color='lime', s=80,
               marker='^', label='Parabolic Up', zorder=5)
    parabolic_down = data[data['Parabolic_Down']]
    ax.scatter(parabolic_down.index, parabolic_down['Price_d3_zscore'], color='darkred', s=80,
               marker='v', label='Parabolic Down', zorder=5)
    ax.set_title('3rd Derivative - Jerk (Parabolic Extremes)', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_signals_with_leverage(data, ticker='TICKER'):
    """
    Plots price with colored line segments.

    Colors:
    - Light Blue: Buy (2x)
    - Lime: Buy (3x)
    - Red: Sell
    - Black: Hold/Default
    """
    data = data.sort_index()

    fig, ax = plt.subplots(figsize=(16, 7))

    # Plot price only
    ax.plot(data.index, data['Close'], color='black', linewidth=1.5, zorder=1)

    # Create line segments by signal
    signal_colors = {
        'Hold': 'black',
        'Buy': 'black',
        'Buy_2x': 'lightblue',
        'Buy_3x': 'lime',
        'Buy (Inflection)': 'black',
        'Buy_2x (Inflection)': 'lightblue',
        'Buy_3x (Inflection)': 'lime',
        'Reduce': 'black',
        'Sell': 'red'
    }

    # Get unique signal periods (where signal changes)
    current_signal = None
    segment_start = 0

    for i in range(len(data)):
        if data['Signal'].iloc[i] != current_signal:
            # Draw previous segment
            if current_signal is not None and i > segment_start:
                segment_data = data.iloc[segment_start:i]
                color = signal_colors.get(current_signal, 'black')
                ax.plot(segment_data.index, segment_data['Close'],
                       color=color, linewidth=3, solid_capstyle='butt', zorder=2)

            current_signal = data['Signal'].iloc[i]
            segment_start = i

    # Draw final segment
    if current_signal is not None and segment_start < len(data):
        segment_data = data.iloc[segment_start:]
        color = signal_colors.get(current_signal, 'black')
        ax.plot(segment_data.index, segment_data['Close'],
               color=color, linewidth=3, solid_capstyle='butt', zorder=2)

    # Simple legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='lightblue', lw=3, label='Buy (2x)'),
        Line2D([0], [0], color='lime', lw=3, label='Buy (3x)'),
        Line2D([0], [0], color='red', lw=3, label='Sell'),
        Line2D([0], [0], color='black', lw=3, label='Hold'),
    ]

    ax.set_title(f'{ticker.upper()} - Trading Signals', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price')
    ax.legend(handles=legend_elements, loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_rsi_with_derivatives(data, ticker='TICKER'):
    """
    Plots RSI + 1st/2nd derivatives for momentum analysis.
    """
    fig, axes = plt.subplots(3, 1, figsize=(16, 10))

    # RSI
    ax = axes[0]
    ax.plot(data.index, data['RSI'], label='RSI (14)', color='purple', linewidth=2)
    ax.axhline(70, color='red', linestyle='--', alpha=0.7, label='Overbought (70)')
    ax.axhline(30, color='green', linestyle='--', alpha=0.7, label='Oversold (30)')
    ax.fill_between(data.index, 30, 70, alpha=0.1, color='gray')
    ax.set_ylabel('RSI')
    ax.set_title(f'{ticker} - RSI & Derivatives', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # RSI 1st Derivative (momentum of momentum)
    ax = axes[1]
    colors = ['green' if x > 0 else 'red' for x in data['RSI_d1']]
    ax.bar(data.index, data['RSI_d1'], color=colors, alpha=0.6)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel("RSI's 1st Derivative")
    ax.set_title('RSI Momentum - Rate of Change', fontsize=12)
    ax.grid(True, alpha=0.3)

    # RSI 2nd Derivative (acceleration of momentum)
    ax = axes[2]
    colors = ['darkgreen' if x > 0 else 'darkred' for x in data['RSI_d2']]
    ax.bar(data.index, data['RSI_d2'], color=colors, alpha=0.6)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_ylabel("RSI's 2nd Derivative")
    ax.set_xlabel('Date')
    ax.set_title('RSI Acceleration - Momentum Change', fontsize=12)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def plot_cumulative_returns(data, ticker):
    """
    Cumulative returns: strategy vs. buy-and-hold.
    """
    if 'BuyHold' not in data.columns or 'Strat_ret' not in data.columns:
        print("Missing return columns.")
        return

    plt.figure(figsize=(16, 6))
    plt.plot(data.index, data['BuyHold'], label='Buy & Hold', color='blue', linewidth=2)
    plt.plot(data.index, data['Strat_ret'], label='Calculus Strategy', color='green', linewidth=2)
    plt.title(f'Cumulative Returns - {ticker.upper()}', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_signals_minimal(data, ticker='TICKER'):
    """
    Ultra-clean: only shows signal CHANGES, not continuous holds.
    Plots markers only at the exact moment signal changes.
    """
    data = data.sort_index()

    fig, ax = plt.subplots(figsize=(16, 7))

    # Plot price and EMAs
    ax.plot(data.index, data['Close'], label='Close', color='black', linewidth=1.5, zorder=1)
    ax.plot(data.index, data['EMA50'], label='EMA50', color='blue', linestyle='--', alpha=0.5)
    ax.plot(data.index, data['EMA200'], label='EMA200', color='red', linestyle='--', alpha=0.5)

    # Find signal changes only
    signal_changes = data[data['Signal'] != data['Signal'].shift()]

    # Plot only changes as markers
    buy_signals = signal_changes[signal_changes['Signal'].str.contains('Buy', case=False, na=False)]
    sell_signals = signal_changes[signal_changes['Signal'].str.contains('Sell', case=False, na=False)]
    reduce_signals = signal_changes[signal_changes['Signal'].str.contains('Reduce', case=False, na=False)]

    # Buy markers (color by leverage)
    buy_1x = buy_signals[~buy_signals['Signal'].str.contains('2x|3x', case=False, na=False)]
    buy_2x = buy_signals[buy_signals['Signal'].str.contains('2x', case=False, na=False)]
    buy_3x = buy_signals[buy_signals['Signal'].str.contains('3x', case=False, na=False)]

    ax.scatter(buy_1x.index, buy_1x['Close'], color='green', s=150, marker='^',
               label='Buy (1x)', zorder=5, edgecolors='darkgreen', linewidth=1)
    ax.scatter(buy_2x.index, buy_2x['Close'], color='teal', s=180, marker='^',
               label='Buy (2x)', zorder=5, edgecolors='darkcyan', linewidth=1)
    ax.scatter(buy_3x.index, buy_3x['Close'], color='lime', s=210, marker='^',
               label='Buy (3x)', zorder=5, edgecolors='darkgreen', linewidth=1)

    ax.scatter(reduce_signals.index, reduce_signals['Close'], color='orange', s=150, marker='v',
               label='Reduce', zorder=5, edgecolors='darkorange', linewidth=1)
    ax.scatter(sell_signals.index, sell_signals['Close'], color='red', s=150, marker='v',
               label='Sell', zorder=5, edgecolors='darkred', linewidth=1)

    ax.set_title(f'{ticker.upper()} - Trading Signals (Entry & Exits Only)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Load Data and Generate Graphs and call functions

In [9]:
# ============================================================================
# SECTION 7: DATA LOADING & ORCHESTRATION (WITH ALL UPDATES)
# ============================================================================

def load_data(ticker, start, end, next_close=None):
    """
    Loads historical OHLCV data with buffer for indicator calculation.
    """
    buffer = 300
    start_date = datetime.strptime(start, "%Y-%m-%d")
    adjusted_start = (start_date - BDay(buffer)).date()

    df = yf.download(
        ticker,
        start=adjusted_start,
        end=end,
        progress=False,
        auto_adjust=False
    )

    df.columns = df.columns.get_level_values(0)
    df = df.sort_index()

    if next_close is not None:
        last_date = df.index[-1]
        next_day = last_date + BDay(1)
        df.loc[next_day, 'Close'] = float(next_close)
        df.loc[next_day, ['Open', 'High', 'Low', 'Adj Close', 'Volume']] = float(next_close), float(next_close), float(next_close), float(next_close), 0

    return df, start_date


def parse_date(date_str, default):
    """
    Parses date string in MM/DD/YY format.
    """
    try:
        return datetime.strptime(date_str.strip(), '%m/%d/%y') if date_str else datetime.strptime(default, '%m/%d/%y')
    except ValueError:
        print("⚠️ Invalid format. Please use MM/DD/YY.")
        return datetime.strptime(default, '%m/%d/%y')


def is_index_ticker(ticker):
    """
    Checks if ticker is an index.
    """
    ticker = ticker.upper()
    if ticker.startswith('^'):
        return True
    index_tickers = ['SPY', 'VOO', 'IVV', 'QQQ', 'DIA', 'IWM', 'VTI', 'VT', 'SPYG', 'SPYV', 'SPLG', 'SCHF', 'SCHB']
    return ticker in index_tickers


def get_user_inputs():
    """
    Interactive input with graph preference upfront.
    - Asks about graphs first
    - Removes cash rate input (uses 3% default)
    - Keeps manual price data option (only if stale)
    """
    from datetime import datetime

    print("\nShow graphs after analysis? (Y/N) [N]: ", end='')
    show_graphs = input().strip().lower() in ['y', 'yes']

    # Get inputs (no cash rate)
    today = datetime.today().strftime('%m/%d/%y')
    ticker = (input("\nTicker [SPY]: ") or 'SPY').upper()
    start = parse_date(input("Start [1/1/20]: "), '01/01/20').strftime('%Y-%m-%d')
    end = parse_date(input("End [today]: "), today).strftime('%Y-%m-%d')

    # Use default cash rate (no input)
    cash = 0.03  # 3% default

    print(f"\n📊 Loading data for {ticker}...")
    temp_df, _ = load_data(ticker, start, end)
    last_date = temp_df.index[-1].date()
    last_price = temp_df['Close'].iloc[-1]

    # Check if data is stale
    today_obj = datetime.today().date()
    days_old = (today_obj - last_date).days

    extra_closes = {}

    # Only prompt if data is actually stale
    if days_old > 0:
        print(f"\n✓ Latest data: {last_date} @ ${last_price:.2f}")
        print(f"⚠️  Data is {days_old} day(s) old")
        print(f"\nAdd more recent price data? (Y/N) [N]: ", end='')
        add_extra = input().strip().lower() in ['y', 'yes']

        if add_extra:
            print("\n📝 Enter prices for newer dates (leave blank to stop)")
            next_date = last_date + BDay(1)

            for i in range(10):
                date_str = next_date.strftime('%m/%d/%y')
                price_input = input(f"  {date_str} close: $").strip()

                if not price_input:
                    break

                try:
                    price = float(price_input)
                    extra_closes[next_date.date()] = price
                    print(f"    ✓ Added: {date_str} = ${price:.2f}")
                    next_date += BDay(1)
                except ValueError:
                    print(f"    ✗ Invalid price. Try again or press Enter to skip.")

            if extra_closes:
                print(f"\n✓ Added {len(extra_closes)} new trading day(s)")
    else:
        # Data is current
        print(f"✓ Latest data: {last_date} @ ${last_price:.2f} (current)")

    # Return show_graphs as well
    return ticker, start, end, cash, extra_closes, show_graphs


def get_signal_explanation(signal, last_row, df):
    """
    Generates detailed explanation of WHY a signal was generated.
    """
    explanation = ""

    # Extract key values
    price = last_row['Close']
    ema50 = last_row['EMA50']
    ema200 = last_row['EMA200']
    rsi = last_row['RSI']
    rsi2 = last_row['RSI2']
    d1 = last_row['Price_d1']
    d2 = last_row['Price_d2']
    d3 = last_row['Price_d3']
    volume_surge = last_row.get('Volume_Surge', False)
    parabolic_up = last_row.get('Parabolic_Up', False)
    parabolic_down = last_row.get('Parabolic_Down', False)

    # Inflection detection
    if len(df) > 1:
        d2_yesterday = df['Price_d2'].iloc[-2]
    else:
        d2_yesterday = 0

    inflection = d2_yesterday < 0 and d2 > 0

    # ===== BUILD EXPLANATION BY SIGNAL TYPE =====

    if 'Buy_3x' in signal:
        explanation = f"""
   🟢 EXTREME BUY SIGNAL (3x Leverage)

   Trend Analysis:
   • EMA50 (${ema50:.2f}) is ABOVE EMA200 (${ema200:.2f}) ✓ Bull trend

   Momentum Detection:
   • Price acceleration (d2 = {d2:.6f}): {"STRONG UP" if d2 > 0.01 else "BUILDING UP"}
   • Price velocity (d1 = {d1:.6f}): {"UPWARD" if d1 > 0 else ""}
   • RSI = {rsi:.1f}: {"OVERSOLD - bottoming" if rsi < 30 else "RISING MOMENTUM"}
   {"• 2nd derivative INFLECTION DETECTED ✓ momentum shift from down to up" if inflection else ""}

   Confluence Factors:
   {"• Volume SURGE detected - strong move confirmation ✓" if volume_surge else ""}
   {"• Parabolic acceleration UP - aggressive momentum ✓" if parabolic_up else ""}
   • Multiple indicators align = HIGH CONFIDENCE

   Signal Strength: MAXIMUM
   → Use 3x leverage, this is an extreme opportunity
   → Execute at tomorrow's open
        """

    elif 'Buy_2x' in signal:
        explanation = f"""
   🟢 STRONG BUY SIGNAL (2x Leverage)

   Trend Analysis:
   • EMA50 (${ema50:.2f}) is ABOVE EMA200 (${ema200:.2f}) ✓ Bull trend

   Momentum Detection:
   • Price acceleration (d2 = {d2:.6f}): {"ACCELERATING UP" if d2 > 0.005 else "TURNING UP"}
   • RSI = {rsi:.1f}: {"OVERSOLD REVERSAL" if rsi < 35 else "CLIMBING"}
   {"• 2nd derivative INFLECTION: acceleration changed from negative to positive ✓" if inflection else ""}

   Confluence Factors:
   {"• Volume SURGE - confidence boost ✓" if volume_surge else ""}
   • Multiple indicators confirm upside
   • Signal strength = STRONG

   → Use 2x leverage, solid entry signal
   → Execute at tomorrow's open
        """

    elif signal == 'Buy' or 'Buy (' in signal:
        explanation = f"""
   🟢 BUY SIGNAL (1x Leverage)

   Trend Analysis:
   • EMA50 (${ema50:.2f}) is ABOVE EMA200 (${ema200:.2f}) ✓ Bull trend

   Momentum Detection:
   • Price moving up (d1 = {d1:.6f}) ✓
   • RSI = {rsi:.1f}: {"Oversold, primed for bounce" if rsi < 35 else "Rising, momentum building"}
   {"• 2nd derivative inflection point detected ✓ turning up" if inflection else "• Confirmed uptrend"}

   Signal Strength: MODERATE
   → Use 1x leverage, standard entry
   → Execute at tomorrow's open
        """

    elif signal == 'Hold':
        explanation = f"""
   🟡 HOLD - NO ACTION

   Trend Analysis:
   • EMA50 (${ema50:.2f}) vs EMA200 (${ema200:.2f}): {"Mixed signals" if abs(ema50-ema200) < price*0.01 else ""}

   Current Status:
   • RSI = {rsi:.1f}: {"Neutral" if 40 < rsi < 60 else ""}
   • Price velocity (d1) = {d1:.6f}: {"Sideways" if abs(d1) < 0.001 else ""}

   → No strong signal to act
   → Maintain current position if any
   → Wait for clearer setup
        """

    elif signal == 'Reduce':
        explanation = f"""
   🟠 REDUCE POSITION - TRIM 50%

   Exhaustion Detected:
   • RSI = {rsi:.1f}: {"OVERBOUGHT - reversal risk" if rsi > 75 else "HIGH - extended move"}
   • Parabolic acceleration (d3): {"EXTREME JERK detected - move unstable ✓" if parabolic_up else ""}
   • Price falling or stalling despite high indicators

   Risk Management:
   • Take profits while available
   • Lock in gains on 50% of position
   • Move stop to breakeven on remaining 50%
   • Let trailing stop catch final moves

   → Reduce at tomorrow's open
   → Exit half, keep half running
        """

    elif signal == 'Sell':
        explanation = f"""
   🔴 SELL - EXIT FULL POSITION

   Reversal Signals:
   • EMA50 (${ema50:.2f}) {"IS BELOW" if ema50 < ema200 else "CROSSED BELOW"} EMA200 (${ema200:.2f})
   • Trend has REVERSED to downside ✓
   • RSI = {rsi:.1f}: {"Declining, momentum weakening" if rsi < 60 else "Still elevated but reversing"}
   {"• Parabolic crash detected (d3 extreme down) ✓" if parabolic_down else ""}

   Exit Rationale:
   • Trend is no longer your friend
   • Risk of sharp downside move
   • Better to be out and wait for new setup

   → Exit full position at tomorrow's open
   → Move to cash
   → Wait for next inflection point to re-enter
        """

    # Add current metrics at bottom
    if not explanation.startswith("\n"):
        explanation = "\n" + explanation

    explanation += f"""
   ─────────────────────────────────────────────
   Current Indicators (Last Day):
   • Price: ${price:.2f}
   • RSI(14): {rsi:.1f}
   • RSI(2): {rsi2:.1f}
   • EMA50: ${ema50:.2f} | EMA200: ${ema200:.2f}
   • Price d1 (velocity): {d1:.6f}
   • Price d2 (acceleration): {d2:.6f}
   • Price d3 (jerk): {d3:.6f}
        """

    return explanation


def print_results(last_signal, metrics, ticker, start, end, df=None):
    """
    Formatted performance summary with detailed signal explanation.
    """
    def format_return(label):
        pct = metrics[f'{label} Cumulative Return']
        multiplier = 1 + pct
        return f"{pct:.1%} ({multiplier:.2f}x)"

    print("\n" + "="*60)
    print(f"🔔 LAST ACTION →  {last_signal.upper()}")
    print("="*60)

    # Get last row data for explanation
    if df is not None:
        last_row = df.iloc[-1]

        # Build explanation based on signal
        explanation = get_signal_explanation(last_signal, last_row, df)
        print(f"\n📊 WHY THIS SIGNAL:{explanation}\n")

    print(f"\n📈 Strategy Performance for {ticker.upper()} | {start} → {end}\n")
    print("🔹 **Strategy (Derivative-Driven)**:\n")
    print(f"   🏆 Sharpe Ratio      : {metrics['Strategy Sharpe Ratio']:.2f}")
    print(f"   📈 Annual Return     : {metrics['Strategy Annualized Return']:.1%}")
    print(f"   📊 Volatility        : {metrics['Strategy Annualized Volatility']:.1%}")
    print(f"   📈 Cumulative Return : {format_return('Strategy')}")
    print(f"   📉 Max Drawdown      : {metrics['Strategy Max Drawdown']:.1%}\n")

    print("🔹 **Buy & Hold Benchmark**:\n")
    print(f"   🏆 Sharpe Ratio      : {metrics['BuyHold Sharpe Ratio']:.2f}")
    print(f"   📈 Annual Return     : {metrics['BuyHold Annualized Return']:.1%}")
    print(f"   📊 Volatility        : {metrics['BuyHold Annualized Volatility']:.1%}")
    print(f"   📈 Cumulative Return : {format_return('BuyHold')}")
    print(f"   📉 Max Drawdown      : {metrics['BuyHold Max Drawdown']:.1%}\n")


def run_strategy(ticker, start, end, cash=0.03, extra_closes=None, verbose=True):
    """
    Main strategy execution pipeline.
    Now properly integrates manual price data and recalculates all indicators.
    """
    print(f"\n⏳ Loading data for {ticker}...")
    df, start_date = load_data(ticker, start, end)

    print(f"📊 Calculating calculus indicators...")
    df = add_calculus_indicators(df)

    # ===== INTEGRATE MANUAL PRICE DATA =====
    if extra_closes:
        print(f"\n📝 Integrating {len(extra_closes)} manual price(s)...")
        for d, price in extra_closes.items():
            date_obj = pd.to_datetime(d)

            # Add to dataframe
            if date_obj not in df.index:
                # Create new row with just the close price
                df.loc[date_obj, 'Close'] = price
                df.loc[date_obj, 'Open'] = price
                df.loc[date_obj, 'High'] = price
                df.loc[date_obj, 'Low'] = price
                df.loc[date_obj, 'Adj Close'] = price
                df.loc[date_obj, 'Volume'] = 0
            else:
                # Update existing row if it exists
                df.loc[date_obj, 'Close'] = price

        # Sort by date
        df = df.sort_index()

        # ===== RECALCULATE ALL INDICATORS =====
        print(f"🔄 Recalculating all indicators with new data...")
        df = add_calculus_indicators(df)
    df = apply_derivative_signal_logic(df, start_date, is_index_ticker(ticker))
    df = calc_leveraged_returns(df, cash)
    metrics = calculate_metrics(df, risk_free_rate=cash)
    last_signal = df['Signal'].iloc[-1]
    if verbose:
        print_results(last_signal, metrics, ticker, start, end, df=df)

    return df, metrics

### Main

In [10]:
def main():
    ticker, start, end, cash, extra_closes, show_graphs = get_user_inputs()
    df, metrics = run_strategy(ticker, start, end, cash, extra_closes=extra_closes)
    if show_graphs:
        print("\n📊 Generating graphs...")
        plot_derivatives(df, ticker)              # Price + 1st/2nd/3rd derivatives
        plot_signals_with_leverage(df, ticker)    # Colored line signals
        plot_rsi_with_derivatives(df, ticker)     # RSI analysis
        plot_cumulative_returns(df, ticker)       # Returns

### RUN:

In [14]:
if __name__ == '__main__':
    main()


Show graphs after analysis? (Y/N) [N]: n

Ticker [SPY]: nvda
Start [1/1/20]: 
End [today]: 

📊 Loading data for NVDA...

✓ Latest data: 2026-05-29 @ $211.14
⚠️  Data is 3 day(s) old

Add more recent price data? (Y/N) [N]: y

📝 Enter prices for newer dates (leave blank to stop)
  06/01/26 close: $220
    ✓ Added: 06/01/26 = $220.00
  06/02/26 close: $

✓ Added 1 new trading day(s)

⏳ Loading data for NVDA...
📊 Calculating calculus indicators...

📝 Integrating 1 manual price(s)...
🔄 Recalculating all indicators with new data...

🔔 LAST ACTION →  BUY_3X

📊 WHY THIS SIGNAL:
   🟢 EXTREME BUY SIGNAL (3x Leverage)

   Trend Analysis:
   • EMA50 ($205.26) is ABOVE EMA200 ($185.08) ✓ Bull trend

   Momentum Detection:
   • Price acceleration (d2 = 1.589998): STRONG UP
   • Price velocity (d1 = 2.466665): UPWARD
   • RSI = 57.3: RISING MOMENTUM
   

   Confluence Factors:
   
   • Parabolic acceleration UP - aggressive momentum ✓
   • Multiple indicators align = HIGH CONFIDENCE

   Signal Stren